# Конспект. Модуль 10: Эмбеддинги и нейросетевые рекомендации

**Курс:** Мини-курс RecSys (13 модулей)
**Модуль:** 10 из 13 — «Эмбеддинги и нейросетевые рекомендации»
**Цель модуля:** понять, что современные нейросетевые рекомендательные системы — не смена парадигмы по отношению к матричной факторизации (Модуль 5), а её **естественное расширение**: та же идея компактных латентных векторов, только (1) обучаемых более гибкой, нелинейной функцией и (2) допускающих на входе произвольные признаки, а не только ID пользователя/товара.

**Важное методологическое примечание:** полноценная реализация моделей из этого модуля требует PyTorch (`nn.Module`, `autograd`) — если эта тема ещё не пройдена по общему плану (раздел «PyTorch» в списке «НЕ ИЗУЧЕНО»), здесь достаточно понять принципы **концептуально и математически** (все числовые примеры ниже, как и в предыдущих модулях, посчитаны и проверены выполнением реального кода), а к практической реализации (10.5) вернуться после освоения основ PyTorch.

## 10.1 От матричной факторизации к эмбеддингам

### 10.1.1 Что уже было эмбеддингом, даже если это слово не звучало

В Модуле 5.2 мы уже вводили латентные векторы `p_u` и `q_i`. По современной терминологии это уже полноценные **эмбеддинги** — плотные векторные представления в низкоразмерном пространстве. Единственное, чем ALS/Funk SVD (Модуль 5) отличаются от «настоящих» нейросетевых эмбеддингов — это **как именно** они обучаются и **как именно** комбинируются для получения финального score.

### 10.1.2 Ключевое ограничение: скалярное произведение — линейная функция взаимодействия

`pred(u,i) = p_u · q_i` — это **билинейная** функция относительно `p_u` и `q_i`. У линейных функций взаимодействия есть фундаментальное ограничение выразительности: существуют закономерности, которые **никакая** билинейная форма не способна выучить точно, вне зависимости от размерности эмбеддингов, — потому что сама структура функции (взвешенная сумма произведений координат) недостаточно богата.

### 10.1.3 Полный проверенный численный пример — XOR как предельный случай

Рассмотрим предельно простую, но показательную закономерность: пусть `a` и `b` — бинарные индикаторы (например, «у пользователя есть признак X» и «у товара есть признак Y»), и истинная релевантность равна **исключающему ИЛИ**:

In [ ]:
(a=0, b=0) -> relevance = 0
(a=0, b=1) -> relevance = 1
(a=1, b=0) -> relevance = 1
(a=1, b=1) -> relevance = 0

**Наилучшая возможная линейная модель** (`pred = w1·a + w2·b + bias`, обученная методом наименьших квадратов — предел того, что вообще способна выразить любая линейная комбинация признаков, включая скалярное произведение эмбеддингов, построенных из этих признаков):

In [ ]:
Оптимальные веса дают одинаковое предсказание для ВСЕХ 4 точек: pred = [0.5, 0.5, 0.5, 0.5]
MSE = 0.2500  — и это теоретический ПРЕДЕЛ, ниже которого линейная модель опуститься не может

Это не ошибка подбора весов — это математически доказуемый факт: XOR **линейно неразделим**, и никакая линейная (в том числе билинейная — то есть основанная на скалярном произведении) модель не может выучить эту закономерность точно, каким бы способом веса ни подбирались.

**Малая нейросеть (2 скрытых нейрона, нелинейная активация `tanh`)** с теми же 4 обучающими примерами:

In [ ]:
Предсказания после обучения: [0.0003, 0.9999, 0.9996, 0.0003]
MSE = 0.000000  — практически точное решение!

**Обученные веса (для понимания механики, не для запоминания):**

In [ ]:
W1 (вход->скрытый слой) = [[0.9063, 1.4906], [1.0761, 2.1480]]
b1 (смещения скрытого слоя) = [-1.4666, -0.3545]
W2 (скрытый->выход) = [-1.2592, 1.2930]
b2 (смещение выхода) = [-0.6916]

### 10.1.4 Почему это не абстрактная математическая забава — прямая связь с реальными паттернами в RecSys

Закономерности типа XOR — не искусственная редкость. Пример из реальной практики: «пользователь любит длинные видео **ИЛИ** короткие эмоциональные ролики, но конкретно не любит видео средней длины» — это структурно похожий на XOR немонотонный паттерн (не «чем больше похоже на A, тем лучше», а «хорошо на двух разных полюсах, плохо в промежутке»), который скалярное произведение линейных эмбеддингов принципиально не может выучить, а MLP-слой поверх тех же эмбеддингов — может, именно за счёт нелинейности. Это именно тот аргумент, который приводился в статье He et al., «Neural Collaborative Filtering» (2017) как обоснование замены `p_u·q_i` на обучаемую MLP-функцию: `pred(u,i) = MLP(p_u, q_i)`.

## 10.2 Item2Vec

### 10.2.1 Аналогия с Word2Vec

Классический Word2Vec (skip-gram) учится предсказывать соседние слова по контексту, в результате чего слова, встречающиеся в похожих контекстах, получают близкие векторы. Item2Vec переносит эту идею на товары: если «корзина покупок» (или сессия просмотра) — это «предложение», то «товары» внутри неё — это «слова». Модель учится предсказывать, что товары `i` и `j`, оказавшиеся в одной корзине, — «соседи», в результате чего товары с похожими паттернами совместного потребления (даже если у них нет ничего общего по контентным признакам, Модуль 6!) получают близкие эмбеддинги.

### 10.2.2 Функция потерь skip-gram с negative sampling

In [ ]:
L = -ln(σ(v_i · v_j)) - Σ_k ln(σ(-v_i · v_k))

где `(i,j)` — позитивная пара (товары из одной корзины), `k` — негативные сэмплы (случайно выбранные товары, **не** встречавшиеся в этой корзине). Обратите внимание на структуру — это **точно та же самая** математическая конструкция, что и BPR (Модуль 5.6.2): максимизировать сходство позитивной пары, минимизировать сходство с негативным сэмплом, через сигмоиду и логарифмическую функцию потерь.

### 10.2.3 Полный проверенный численный пример — один шаг обучения

«Корзина» содержит `I1, I2, I4` (куплены вместе). `I5` не было в этой корзине — берём его как негативный сэмпл. Начальные (иллюстративные) векторы:

In [ ]:
v_I1 = [0.2, -0.1]   v_I2 = [0.3, 0.4]   v_I5 = [-0.2, 0.5]

**Шаг 1. Считаем текущие score:**

In [ ]:
score(I1,I2) [позитив]  = v_I1·v_I2 = 0.0200,  σ(0.0200) = 0.5050
score(I1,I5) [негатив]  = v_I1·v_I5 = -0.0900, σ(-0.0900) = 0.4775

**Шаг 2. Градиенты и обновление (`lr=0.1`):**

In [ ]:
v_I1: [0.2000, -0.1000] -> [0.2244, -0.1041]
v_I2: [0.3000,  0.4000] -> [0.3099,  0.3950]
v_I5: [-0.2000, 0.5000] -> [-0.2096, 0.5048]

**Шаг 3. Проверяем результат:**

In [ ]:
Новый score(I1,I2) = 0.0284 (было 0.0200)   -> ВЫРОС, как и требовалось (позитивная пара сближается)
Новый score(I1,I5) = -0.0996 (было -0.0900)  -> УПАЛ, как и требовалось (негативная пара отдаляется)

**Итог, за много таких шагов по всей истории корзин:** `I1` и `I2` (часто встречающиеся вместе) постепенно получают близкие векторы, `I1` и `I5` (редко пересекающиеся) — отдаляются. Классический пример результата такого обучения на реальных данных: «пиво» и «чипсы», «подгузники» и «детское питание» оказываются близко в эмбеддинг-пространстве — паттерны совместного потребления, извлечённые полностью автоматически, без единого явного признака «эти товары часто покупают вместе».

## 10.3 Two-Tower модели (DSSM — Deep Structured Semantic Models)

### 10.3.1 Архитектура

Две **независимые** нейросети:
- **User Tower:** вход = ID пользователя + демография + история действий -> выход = вектор `u ∈ R^d`.
- **Item Tower:** вход = ID товара + категория + текстовое описание -> выход = вектор `v ∈ R^d`.

Релевантность: `score(u,i) = u · v` (или косинус). Обратите внимание: сам **финальный** шаг сравнения — по-прежнему скалярное произведение (та же потенциальная ограниченность, что в 10.1.2-10.1.3!) — но каждый из векторов `u` и `v` **сам по себе** уже получен нелинейной функцией от богатого набора исходных признаков, а не хранится как отдельно обучаемый параметр для каждого ID, как в чистом ALS (Модуль 5.5). Это ключевое отличие: Two-Tower может выдать разумный эмбеддинг для **нового** пользователя/товара, для которого известны признаки, но ещё нет истории взаимодействий (частичное решение cold start, дополняющее подходы Модуля 6-7).

### 10.3.2 Полный проверенный численный пример forward pass

**User Tower** (вход: 3 признака -> `Linear(4) -> ReLU -> Linear(2)`):

In [ ]:
user_features = [0.5, -0.3, 0.8]
После скрытого слоя (ReLU): [0.0715, 0.3176, 0.0000, 0.1794]
User embedding u = [-0.3467, -0.2843]

**Item Tower** (вход: 3 признака -> `Linear(4) -> ReLU -> Linear(2)`), для двух товаров-кандидатов:

In [ ]:
Товар 1: item_features = [0.9, 0.1, -0.2]
  После скрытого слоя (ReLU): [0.7924, 0.0331, 0.1330, 0.0000]
  Item embedding v1 = [0.0222, -0.5696]
  Score1 = u · v1 = 0.1543

Товар 2: item2_features = [-0.5, 0.9, 0.3]
  Item embedding v2 = [-0.5329, -0.0104]
  Score2 = u · v2 = 0.1877

**Интерпретация:** на этой (пока необученной, инициализированной случайно — числа приведены только для демонстрации механики forward pass) сети товар 2 получил чуть более высокий score, чем товар 1 — именно так, после полноценного обучения на реальных данных, Two-Tower модель ранжировала бы кандидатов для показа пользователю.

### 10.3.3 Почему именно эта архитектура — основа production Retrieval (важнейший практический вывод модуля)

Ключевое архитектурное преимущество, ради которого Two-Tower стала стандартом де-факто для стадии отбора кандидатов (Retrieval, подробно — Модуль 11.2-11.3): **вычисление User Tower и Item Tower полностью независимо** — у них нет общих слоёв, объединяющих признаки пользователя и товара до самого последнего шага (скалярного произведения). Это позволяет:

1. **Посчитать все эмбеддинги товаров заранее, офлайн** (batch job, аналогично предвычислению item-item сходства в Модуле 4.4.1) — даже если каталог насчитывает миллионы позиций.
2. Сложить их в индекс приближенного поиска ближайших соседей (FAISS/HNSW — подробно Модуль 11.3).
3. В момент реального запроса пользователя нужен **только один** forward pass через User Tower (быстро — один пользователь, не миллион товаров) плюс быстрый ANN-поиск по уже готовому индексу.

**Конкретная иллюстрация выгоды (в духе уже привычных вам расчётов latency budget из предыдущих модулей):** если бы вместо Two-Tower использовалась единая сеть, принимающая на вход **пару** (пользователь, товар) сразу (как в исходной идее NCF, раздел 10.1.4), пришлось бы прогонять полный forward pass через всю сеть **для каждого товара в каталоге отдельно** в момент запроса — при каталоге в 10 миллионов товаров и бюджете в 50 мс на ответ это физически невозможно. Two-Tower переносит основную вычислительную нагрузку в офлайн, оставляя на онлайн-момент лишь один быстрый forward pass и дешёвый ANN-lookup.

## 10.4 Deep Neural Networks for YouTube Recommendations (Covington et al., Google, 2016)

### 10.4.1 Двухстадийная архитектура — первое явное описание в промышленной статье

- **Candidate Generation (Retrieval):** Two-Tower-подобная модель (10.3), отбирает несколько сотен кандидатов из каталога, состоящего из миллионов видео.
- **Ranking:** отдельная, более глубокая и «дорогая» сеть, использующая сотни признаков (пользователь, видео, контекст, история просмотров), применяется только к уже отобранным на первом этапе кандидатам.

### 10.4.2 Почему именно две отдельные модели, а не одна

Это прямая иллюстрация принципа, уже введённого в Модуле 7.2.5 (Cascade Hybrid): кандидатная модель должна быть **очень быстрой**, потому что работает с миллионами товаров, — точность здесь приносится в жертву скорости, лишь бы не упустить действительно релевантные варианты (высокий **Recall**, Модуль 8.2, важнее высокой Precision на этом этапе). Ранжирующая модель, наоборот, может себе позволить быть медленной и сложной — потому что работает уже только с сотнями кандидатов, где важна максимальная точность (**Precision** и позиционно-чувствительные метрики — NDCG, Модуль 8.5, — выходят на первый план).

**Это прямое предвосхищение архитектуры Модуля 11** — по сути, вся оставшаяся структура курса (двухстадийные production-системы) уже полностью описана здесь, на конкретном промышленном примере YouTube, только предстоит формализовать её в общем виде и добавить третий, финальный этап (Re-ranking, бизнес-правила — уже знакомый вам по Модулю 7.2.5 и снова встретится в 11.5).

## 10.5 Практика

### 10.5.1 Реализация простой Two-Tower модели на PyTorch

In [ ]:
import torch
import torch.nn as nn

class UserTower(nn.Module):
    def __init__(self, n_users, embedding_dim=32, hidden_dim=64):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 32)
        )

    def forward(self, user_ids):
        x = self.user_embedding(user_ids)
        return self.mlp(x)

class ItemTower(nn.Module):
    def __init__(self, n_items, n_categories, embedding_dim=32, hidden_dim=64):
        super().__init__()
        self.item_embedding = nn.Embedding(n_items, embedding_dim)
        self.category_embedding = nn.Embedding(n_categories, 8)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim + 8, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 32)
        )

    def forward(self, item_ids, category_ids):
        x = torch.cat([self.item_embedding(item_ids), self.category_embedding(category_ids)], dim=1)
        return self.mlp(x)

# BPR loss (Модуль 5.6) - естественный выбор функции потерь для обучения Two-Tower на implicit-данных
def bpr_loss(user_vec, pos_item_vec, neg_item_vec):
    pos_score = (user_vec * pos_item_vec).sum(dim=1)
    neg_score = (user_vec * neg_item_vec).sum(dim=1)
    return -torch.log(torch.sigmoid(pos_score - neg_score)).mean()

### 10.5.2 Обучение на MovieLens и визуализация

- Обучить Two-Tower модель на MovieLens 1M, используя `bpr_loss` (10.5.1) с негативными сэмплами (случайные фильмы, которые пользователь не оценивал — та же логика, что в Модуле 5.5.4/5.6).
- Визуализировать полученные эмбеддинги товаров через `t-SNE` (`sklearn.manifold.TSNE`) — на реальных данных обычно видно, как фильмы одного жанра образуют визуальные кластеры **без единого explicit указания жанра** в целевой функции обучения (только implicit-взаимодействия) — наглядное доказательство того, что модель сама «открыла» структуру, аналогичную явным контентным признакам из Модуля 6.
- **Сравнение при отсутствии PyTorch-практики (временный вариант):** если PyTorch ещё не пройден, эквивалент этого упражнения — обучить Item2Vec (раздел 10.2) на «корзинах» (сессиях) с помощью `gensim.models.Word2Vec`, что не требует ручной реализации нейросети, и сравнить полученную кластеризацию эмбеддингов с результатом ALS (Модуль 5.7.2).

### 10.5.3 Вопросы для самопроверки

1. В разделе 10.1.3 показано, что MLP решает XOR, а линейная модель — нет. Означает ли это, что нужно **всегда** предпочитать MLP-взаимодействие (10.1.4) простому скалярному произведению (Модуль 5)? Какую цену (вспомните материал предыдущих модулей о вычислительной сложности) вы бы за это заплатили в production?
2. Почему в разделе 10.3.3 подчёркивается, что у User Tower и Item Tower **нет общих слоёв**? Что конкретно сломалось бы в описанной там схеме «предвычислить эмбеддинги товаров офлайн», если бы архитектура требовала совместной обработки пары (user, item) хотя бы на одном промежуточном слое?
3. В разделе 10.2.2 отмечается, что функция потерь Item2Vec математически идентична BPR (Модуль 5.6.2). Если это так, то в чём тогда содержательная (не математическая, а прикладная) разница между «обучить ALS на implicit-матрице взаимодействий» (Модуль 5.5.4) и «обучить Item2Vec на корзинах покупок» (10.2)? Подсказка: подумайте, что именно считается «контекстом» в каждом случае.
4. Опишите своими словами, как принцип Cascade (Модуль 7.2.5) буквально реализован в архитектуре YouTube (10.4.2) — какая модель в этой паре быстрая, а какая точная, и почему их роли не могут поменяться местами.

## Глоссарий модуля 10

| Термин | Короткое определение |
|:---|:---|
| Эмбеддинг | Плотный вектор в низкоразмерном пространстве, представляющий сущность (пользователя, товар, слово) |
| Линейная разделимость | Свойство данных, при котором простая линейная функция может их корректно разделить/предсказать |
| NCF (Neural Collaborative Filtering) | Замена скалярного произведения на обучаемую MLP-функцию взаимодействия |
| Item2Vec | Skip-gram-подобное обучение эмбеддингов товаров на основе co-occurrence в корзинах/сессиях |
| Two-Tower (DSSM) | Архитектура из двух независимых сетей (User Tower, Item Tower), сравниваемых скалярным произведением |
| Candidate Generation | Быстрая стадия отбора кандидатов из полного каталога |
| t-SNE | Метод визуализации многомерных эмбеддингов на 2D-плоскости |

**Связь со следующим модулем:** раздел 10.3.3 и 10.4 уже фактически описали архитектуру, которую Модуль 11 формализует как общий промышленный стандарт — Retrieval (быстрый отбор кандидатов через ANN по предвычисленным эмбеддингам) + Ranking (точная модель, Модуль 9, на отобранных кандидатах) + Re-ranking (бизнес-правила, Модуль 7.2.5). Всё, что нужно для полного понимания production-архитектуры, у вас уже есть по частям — Модуль 11 их окончательно связывает в единую систему.